# 08 머신러닝 예측 모델 (Phase 1 — 2/3)

**40조건** × **RF, XGBoost** | 지표: MAE, RMSE, MAPE, MASE

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cpu
시계열: 165
학습 <= 201730 | 검증: [201731, 201732, 201733]
Best 선정 기준: MAPE


### ① ML 모델 실험

In [2]:
# Phase 1 — ML 2종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_ml_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=ML_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=ML_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_ml_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_ml_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/20 [00:00<?, ?it/s]

Phase1 ML:   0%|          | 0/20 [00:00<?, ?it/s]

완료 | 660 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
1,ML,A,1,XGBoost,2462.732860,3361.175379,99.807025,4.524735
3,ML,A,2,XGBoost,142286.177083,192591.435250,105.969377,3.789674
5,ML,A,3,XGBoost,58266.037150,80798.303933,59.800775,2.536248
7,ML,A,4,XGBoost,22253.233989,32108.340274,96.838858,3.578794
9,ML,B,1,XGBoost,2051.172302,2990.261434,97.434245,2.898510
11,ML,B,4,XGBoost,40878.801764,58713.693148,75.042206,2.771052
12,ML,C,1,RF,1752.037931,2609.719805,116.914783,4.153405
14,ML,C,2,RF,75887.663677,113124.954524,74.104331,3.271271
17,ML,C,4,XGBoost,39584.988681,48773.334549,77.589336,5.061460
19,ML,D,1,XGBoost,2553.850063,3761.929759,129.707018,4.190404



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML    SBC
model                        
RF              114.54  117.6
XGBoost         145.37  105.1
